In [ ]:
# ============================================================
# SmartWarehouse Hybrid Final Code
# H-2 feature engineering + AutoEncoder two-stage strategy
# + pred-lag Stage2 + final grid/meta selection + calibration
# ============================================================

import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")


# ============================================================
# 0. Config
# ============================================================

SEED = 42
TARGET = "avg_delay_minutes_next_30m"
DATA_DIR = "./data"

USE_LOG_TARGET = True
N_FOLDS = 5 # 제출 용으로는 5

USE_AUTOENCODER = True
AE_LATENT_DIM = 32
AE_HIDDEN_DIM = 256
AE_EPOCHS_MAX = 30 # 실험용 15, 제출용 30
AE_ES_PATIENCE = 5
AE_BATCH_SIZE = 4096
AE_LR = 1e-3
AE_WEIGHT_DECAY = 3e-5
AE_DROPOUT = 0.0
AE_DENOISE_STD = 0.0
AE_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ENABLE_SAMPLE_WEIGHT = True
SW_HIGH_Q = 0.90
SW_HIGH_MULT = 1.25
SW_TS_EDGE_MULT = 1.10

USE_STAGE1_RESIDUAL = False
USE_S2_PRED_LAG_ONLY = True

CLIP_PRED_MIN = 0.0
FINAL_CLIP_Q_GRID = [0.991, 0.992, 0.993, 0.994, 0.995, 0.996, 0.997, 0.998]
FINAL_ALPHA_STEP = 0.005
JOINT_P_GRID = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0]

ENABLE_TIMESLOT_ALPHA_SEARCH = True
TIMESLOT_ALPHA_STEP = 0.02
TIMESLOT_ALPHA_SMOOTH_TO_GLOBAL = 0.20

ENABLE_OOF_META_STACK = True
META_ALPHAS = np.logspace(-4, 4, 41)

AUTO_SELECT_FINAL_METHOD = True

GLOBAL_CALIBRATION_SEARCH = True


# ============================================================
# 1. Utility
# ============================================================

def elapsed(start):
    s = int(time.time() - start)
    return f"{s // 60}m {s % 60:02d}s"


def section(title):
    print(f"\n{'=' * 70}\n  {title}\n{'=' * 70}")


def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)


def to_train_target(y):
    return np.log1p(y) if USE_LOG_TARGET else y


def from_train_pred(p):
    return np.expm1(p) if USE_LOG_TARGET else p


def apply_scale_bias_clip(pred, scale, bias, clip_q):
    pred = pred * scale + bias
    pred = np.clip(pred, 0, None)

    upper = None
    if clip_q is not None:
        upper = np.percentile(pred, clip_q)
        pred = np.clip(pred, 0, upper)

    return pred, upper


def search_scale_bias_clip(oof_pred, y_true):
    scales = np.arange(0.94, 1.031, 0.005)
    biases = np.arange(-1.0, 1.01, 0.25)
    clip_qs = [97.5, 98.0, 98.5, 99.0, 99.5, None]

    rows = []
    best_score = 999
    best_params = None

    for scale in scales:
        for bias in biases:
            for clip_q in clip_qs:
                pred, upper = apply_scale_bias_clip(oof_pred, scale, bias, clip_q)
                score = mae(y_true, pred)

                rows.append({
                    "scale": scale,
                    "bias": bias,
                    "clip_q": clip_q,
                    "clip_upper": upper,
                    "mae": score
                })

                if score < best_score:
                    best_score = score
                    best_params = {
                        "scale": float(scale),
                        "bias": float(bias),
                        "clip_q": clip_q,
                        "clip_upper": upper
                    }

    result_df = pd.DataFrame(rows).sort_values("mae")

    print("\n===== Global Calibration Search =====")
    print("best params:", best_params)
    print("best calibration MAE:", best_score)
    display(result_df.head(20))

    return best_params, best_score, result_df


def _ensemble_pred(oof_by_model, maes_by_model, models, p):
    weights = {m: 1.0 / (maes_by_model[m] ** p) for m in models}
    weight_sum = sum(weights.values())

    out = np.zeros_like(next(iter(oof_by_model.values())))
    for m in models:
        out += weights[m] * oof_by_model[m]

    return out / weight_sum, weights, weight_sum


def powerset_models():
    return [
        ["lgb"], ["xgb"], ["cat"],
        ["lgb", "xgb"], ["lgb", "cat"], ["xgb", "cat"],
        ["lgb", "xgb", "cat"]
    ]


# ============================================================
# 2. Load Data
# ============================================================

train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
sample_path = os.path.join(DATA_DIR, "sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("train shape:", train.shape)
print("test shape :", test.shape)


# ============================================================
# 3. H-2 style feature engineering
# ============================================================

def add_features_group_aware_v2(df):
    df = df.copy()
    group_key = "scenario_id"
    new_features = {}

    if "robot_active" in df.columns and "robot_idle" in df.columns:
        new_features["active_idle_ratio"] = df["robot_active"] / (df["robot_idle"] + 1)

    if "order_inflow_15m" in df.columns and "robot_active" in df.columns:
        new_features["orders_per_robot"] = df["order_inflow_15m"] / (df["robot_active"] + 1)

    if "charge_queue_length" in df.columns and "robot_charging" in df.columns:
        new_features["charge_pressure"] = df["charge_queue_length"] / (df["robot_charging"] + 1)

    if "congestion_score" in df.columns and "robot_active" in df.columns:
        new_features["congestion_robot"] = df["congestion_score"] * df["robot_active"]

    if "battery_std" in df.columns and "robot_charging" in df.columns:
        new_features["battery_charge_interaction"] = df["battery_std"] * df["robot_charging"]

    group_cols = [
        "order_inflow_15m", "robot_active", "congestion_score",
        "battery_std", "robot_charging", "charge_queue_length",
        "max_zone_density", "sku_concentration", "manual_override_ratio",
        "pack_utilization", "avg_trip_distance", "robot_utilization",
        "robot_idle", "battery_mean", "outbound_truck_wait_min"
    ]

    for col in group_cols:
        if col in df.columns:
            grp = df.groupby(group_key)[col]

            grp_mean = grp.transform("mean")
            grp_std = grp.transform("std").fillna(0)
            grp_min = grp.transform("min")
            grp_max = grp.transform("max")
            grp_median = grp.transform("median")

            new_features[f"{col}_grp_mean"] = grp_mean
            new_features[f"{col}_grp_std"] = grp_std
            new_features[f"{col}_grp_min"] = grp_min
            new_features[f"{col}_grp_max"] = grp_max
            new_features[f"{col}_grp_median"] = grp_median
            new_features[f"{col}_diff_mean"] = df[col] - grp_mean
            new_features[f"{col}_diff_median"] = df[col] - grp_median
            new_features[f"{col}_ratio_mean"] = df[col] / (grp_mean + 1e-6)
            new_features[f"{col}_zscore"] = (df[col] - grp_mean) / (grp_std + 1e-6)
            new_features[f"{col}_rank"] = grp.rank(method="average")
            new_features[f"{col}_pct_rank"] = grp.rank(pct=True)
            new_features[f"{col}_range_pos"] = (df[col] - grp_min) / (grp_max - grp_min + 1e-6)

    interaction_pairs = [
        ("order_inflow_15m", "max_zone_density"),
        ("order_inflow_15m", "robot_active"),
        ("order_inflow_15m", "congestion_score"),
        ("order_inflow_15m", "sku_concentration"),
        ("robot_active", "congestion_score"),
        ("robot_active", "manual_override_ratio"),
        ("robot_utilization", "congestion_score"),
        ("robot_utilization", "battery_std"),
        ("battery_std", "congestion_score"),
        ("charge_queue_length", "robot_charging"),
        ("pack_utilization", "sku_concentration"),
        ("avg_trip_distance", "order_inflow_15m")
    ]

    for a, b in interaction_pairs:
        if a in df.columns and b in df.columns:
            new_features[f"{a}_x_{b}"] = df[a] * df[b]
            new_features[f"{a}_div_{b}"] = df[a] / (df[b] + 1e-6)

    feature_df = pd.DataFrame(new_features, index=df.index)
    feature_df = feature_df.replace([np.inf, -np.inf], np.nan)

    df = pd.concat([df, feature_df], axis=1)
    df = df.copy()

    return df


def add_time_features_v2(df):
    df = df.copy()

    if "scenario_id" not in df.columns:
        return df

    df["_orig_order"] = np.arange(len(df))

    possible_time_cols = [
        "timestamp", "time", "datetime", "minute", "timestep",
        "step", "slot", "time_idx"
    ]

    time_col = None
    for col in possible_time_cols:
        if col in df.columns:
            time_col = col
            break

    if time_col is not None:
        df = df.sort_values(["scenario_id", time_col]).copy()
        print("time feature 기준 컬럼:", time_col)
    else:
        df = df.sort_values(["scenario_id", "_orig_order"]).copy()
        print("time feature 기준: scenario_id 내부 원본 행 순서")

    key_cols = [
        "order_inflow_15m",
        "robot_active",
        "robot_idle",
        "robot_charging",
        "congestion_score",
        "max_zone_density",
        "charge_queue_length",
        "battery_std",
        "battery_mean",
        "sku_concentration",
        "manual_override_ratio",
        "pack_utilization",
        "avg_trip_distance",
        "robot_utilization",
        "outbound_truck_wait_min"
    ]

    lags = [1, 2, 3, 5, 7]
    rolls = [3, 5, 7, 10]

    new_features = {}

    for col in key_cols:
        if col not in df.columns:
            continue

        grp = df.groupby("scenario_id")[col]

        for lag in lags:
            lag_val = grp.shift(lag)
            new_features[f"{col}_lag{lag}"] = lag_val
            new_features[f"{col}_diff_lag{lag}"] = df[col] - lag_val

        shifted = grp.shift(1)

        for window in rolls:
            new_features[f"{col}_roll{window}_mean"] = (
                shifted.groupby(df["scenario_id"])
                .rolling(window, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )

            new_features[f"{col}_roll{window}_std"] = (
                shifted.groupby(df["scenario_id"])
                .rolling(window, min_periods=1)
                .std()
                .reset_index(level=0, drop=True)
                .fillna(0)
            )

        prev = grp.shift(1)
        new_features[f"{col}_pct_change1"] = (df[col] - prev) / (prev.abs() + 1e-6)

    feature_df = pd.DataFrame(new_features, index=df.index)
    feature_df = feature_df.replace([np.inf, -np.inf], np.nan)

    df = pd.concat([df, feature_df], axis=1)

    df = df.sort_values("_orig_order").drop(columns=["_orig_order"])
    df = df.copy()

    return df


def add_auto_strategy_features(df):
    df = df.copy()

    if "timeslot" not in df.columns:
        df["timeslot"] = df.groupby("scenario_id").cumcount()

    if "robot_active" in df.columns and "robot_total" in df.columns:
        df["robot_efficiency"] = df["robot_active"] / (df["robot_total"] + 1e-6)

    if "robot_total" in df.columns and "floor_area_sqm" in df.columns:
        df["robot_density"] = df["robot_total"] / (df["floor_area_sqm"] + 1e-6)

    if "order_inflow_15m" in df.columns and "pack_station_count" in df.columns:
        df["order_per_station"] = df["order_inflow_15m"] / (df["pack_station_count"] + 1e-6)
        df["cumulative_orders"] = df.groupby("scenario_id")["order_inflow_15m"].cumsum()
        df["order_pressure"] = df["cumulative_orders"] / (df["pack_station_count"] + 1e-6)

    if "robot_active" in df.columns and "pack_station_count" in df.columns:
        df["robot_per_station"] = df["robot_active"] / (df["pack_station_count"] + 1e-6)

    if "congestion_score" in df.columns and "robot_efficiency" in df.columns:
        df["risk_index"] = df["congestion_score"] * (1 - df["robot_efficiency"])

    if "order_per_station" in df.columns and "congestion_score" in df.columns:
        df["bottle_neck"] = df["order_per_station"] * df["congestion_score"]

    if "low_battery_ratio" in df.columns and "robot_total" in df.columns:
        df["battery_risk"] = df["low_battery_ratio"] * df["robot_total"]

    if "battery_mean" in df.columns and "battery_std" in df.columns:
        df["battery_cv"] = df["battery_std"] / (df["battery_mean"] + 1e-6)

    return df


def encode_object_columns(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()

    common_cols = train_df.columns.intersection(test_df.columns)

    for col in common_cols:
        if train_df[col].dtype == "object" or test_df[col].dtype == "object":
            combined = pd.concat([train_df[col], test_df[col]], axis=0)
            combined = combined.astype(str).fillna("missing")
            codes, _ = pd.factorize(combined)

            train_df[col] = codes[:len(train_df)]
            test_df[col] = codes[len(train_df):]

    return train_df, test_df


train = add_auto_strategy_features(train)
test = add_auto_strategy_features(test)

train = add_features_group_aware_v2(train)
test = add_features_group_aware_v2(test)

train = add_time_features_v2(train)
test = add_time_features_v2(test)

train, test = encode_object_columns(train, test)

print("after feature train shape:", train.shape)
print("after feature test shape :", test.shape)


# ============================================================
# 4. Target Encoding
# ============================================================

ID_COLS = ["ID", "layout_id", "scenario_id"]

try:
    from sklearn.model_selection import StratifiedGroupKFold

    scenario_mean = train.groupby("scenario_id")[TARGET].transform("mean")
    y_strat = pd.qcut(scenario_mean, q=10, labels=False, duplicates="drop")
    y_strat = pd.Series(y_strat).fillna(0).astype(int).values

    kf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    kf_y = y_strat
    print("CV: StratifiedGroupKFold")
except Exception as e:
    kf = GroupKFold(n_splits=N_FOLDS)
    kf_y = train[TARGET].values
    print("CV: GroupKFold", e)

groups = train["scenario_id"].values
global_mean = float(train[TARGET].mean())

TE_COLS = [c for c in ["layout_id", "timeslot"] if c in train.columns]

def apply_target_encoding(train_df, test_df, col):
    te_col = f"{col}_te"
    train_df[te_col] = np.nan

    for tr_idx, va_idx in kf.split(train_df, kf_y, groups=groups):
        tr_key = train_df.iloc[tr_idx][col]
        va_key = train_df.iloc[va_idx][col]
        tr_target = train_df.iloc[tr_idx][TARGET]

        stats = pd.DataFrame({
            "key": tr_key.values,
            "target": tr_target.values
        }).groupby("key")["target"].agg(["mean", "count"])

        smoothing = 20
        smooth = (stats["count"] * stats["mean"] + smoothing * global_mean) / (stats["count"] + smoothing)

        train_df.loc[train_df.index[va_idx], te_col] = va_key.map(smooth).fillna(global_mean)

    stats_full = train_df.groupby(col)[TARGET].agg(["mean", "count"])
    smooth_full = (stats_full["count"] * stats_full["mean"] + 20 * global_mean) / (stats_full["count"] + 20)
    test_df[te_col] = test_df[col].map(smooth_full).fillna(global_mean)

    return train_df, test_df


for col in TE_COLS:
    train, test = apply_target_encoding(train, test, col)


# ============================================================
# 5. AutoEncoder
# ============================================================

class TabularAutoEncoder(nn.Module):
    def __init__(self, n_in, hidden, latent, dropout=0.0):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(n_in, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, latent)
        )
        self.dec = nn.Sequential(
            nn.Linear(latent, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_in)
        )

    def forward(self, x):
        z = self.enc(x)
        x_hat = self.dec(z)
        return x_hat, z


def ae_prepare_matrix(df, cols, medians):
    X = df[cols].to_numpy(dtype=np.float64)
    for j, col in enumerate(cols):
        m = float(medians[col])
        v = X[:, j]
        v[~np.isfinite(v)] = m
        v[np.isnan(v)] = m
    return X


def ae_train_fold(X_tr, X_va, device, seed):
    torch.manual_seed(seed)

    n_in = X_tr.shape[1]
    model = TabularAutoEncoder(n_in, AE_HIDDEN_DIM, AE_LATENT_DIM, AE_DROPOUT).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=AE_LR, weight_decay=AE_WEIGHT_DECAY)
    scaler = StandardScaler()

    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)

    tr_t = torch.from_numpy(X_tr_s).float().to(device)
    va_t = torch.from_numpy(X_va_s).float().to(device)

    n = tr_t.shape[0]
    best_val = float("inf")
    best_state = None
    patience = 0

    for epoch in range(AE_EPOCHS_MAX):
        model.train()
        perm = torch.randperm(n, device=device)

        for i in range(0, n, AE_BATCH_SIZE):
            idx = perm[i:i + AE_BATCH_SIZE]
            xb = tr_t[idx]

            if AE_DENOISE_STD > 0:
                xb_in = xb + torch.randn_like(xb) * AE_DENOISE_STD
            else:
                xb_in = xb

            recon, _ = model(xb_in)
            loss = ((recon - xb) ** 2).mean()

            opt.zero_grad()
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            recon_va, _ = model(va_t)
            val_loss = ((recon_va - va_t) ** 2).mean().item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1

        if patience >= AE_ES_PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        _, z_va = model(va_t)

    return scaler, model.cpu(), z_va.cpu().numpy()


def ae_encode_matrix(X_raw, scaler, model, device):
    X_s = scaler.transform(X_raw)
    t = torch.from_numpy(X_s).float().to(device)
    model = model.to(device)
    model.eval()

    zs = []
    with torch.no_grad():
        for i in range(0, len(t), AE_BATCH_SIZE):
            _, z = model(t[i:i + AE_BATCH_SIZE])
            zs.append(z.cpu().numpy())

    model.cpu()
    return np.vstack(zs)


# ============================================================
# 6. Stage 1 feature columns
# ============================================================

base_exclude = ID_COLS + [TARGET]
feature_cols_s1_base = [c for c in train.columns if c not in base_exclude]

ae_input_cols = [
    c for c in feature_cols_s1_base
    if pd.api.types.is_numeric_dtype(train[c])
]

AE_COLS = [f"ae_z{i}" for i in range(AE_LATENT_DIM)] if USE_AUTOENCODER else []

for c in AE_COLS:
    train[c] = 0.0
    test[c] = 0.0

feature_cols_s1 = feature_cols_s1_base + AE_COLS

y_raw = train[TARGET].values
y_all = to_train_target(y_raw)


def build_sample_weight(df):
    w = np.ones(len(df), dtype=np.float64)

    if not ENABLE_SAMPLE_WEIGHT:
        return w

    y = df[TARGET].values
    high_thr = np.quantile(y, 0.90)
    w[y >= high_thr] *= SW_HIGH_MULT

    if "timeslot" in df.columns:
        ts = df["timeslot"].values
        edge = (ts <= 2) | (ts >= 22)
        w[edge] *= SW_TS_EDGE_MULT

    return w


sw_all = build_sample_weight(train)

print("Stage1 features:", len(feature_cols_s1))
print("AE cols:", len(AE_COLS))
print("sample weight mean/max:", sw_all.mean(), sw_all.max())


# ============================================================
# 7. Stage 1 models
# ============================================================

lgb_params_s1 = dict(
    objective="regression_l1",
    n_estimators=25000,
    learning_rate=0.01,
    max_depth=-1,
    num_leaves=2047,
    min_child_samples=60,
    subsample=0.75,
    subsample_freq=1,
    colsample_bytree=0.5,
    reg_alpha=0.3,
    reg_lambda=5.0,
    random_state=SEED,
    verbose=-1
)

xgb_params_s1 = dict(
    objective="reg:absoluteerror",
    n_estimators=20000,
    learning_rate=0.015,
    max_depth=10,
    subsample=0.75,
    colsample_bytree=0.5,
    colsample_bynode=0.5,
    reg_alpha=0.3,
    reg_lambda=3.0,
    random_state=SEED,
    tree_method="hist",
    eval_metric="mae",
    early_stopping_rounds=500,
    verbosity=0
)

cat_params_s1 = dict(
    iterations=20000,
    learning_rate=0.015,
    depth=10,
    l2_leaf_reg=5.0,
    bootstrap_type="MVS",
    subsample=0.75,
    colsample_bylevel=0.5,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=SEED,
    task_type="CPU",
    early_stopping_rounds=500
)


section("Stage 1 - LGB/XGB/CAT + AutoEncoder")
t0 = time.time()

oof_s1_lgb = np.zeros(len(train))
oof_s1_xgb = np.zeros(len(train))
oof_s1_cat = np.zeros(len(train))

models_s1_lgb = []
models_s1_xgb = []
models_s1_cat = []
ae_fold_artifacts = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(train, kf_y, groups=groups), 1):
    print(f"\n--- Stage1 Fold {fold} ---")

    tr_df = train.iloc[tr_idx].copy()
    va_df = train.iloc[va_idx].copy()

    if USE_AUTOENCODER:
        med_ae = tr_df[ae_input_cols].median()
        X_ae_tr = ae_prepare_matrix(tr_df, ae_input_cols, med_ae)
        X_ae_va = ae_prepare_matrix(va_df, ae_input_cols, med_ae)

        sc_ae, model_ae, z_va = ae_train_fold(X_ae_tr, X_ae_va, AE_DEVICE, SEED + fold)
        train.loc[train.index[va_idx], AE_COLS] = z_va
        va_df[AE_COLS] = z_va
        ae_fold_artifacts.append((med_ae, sc_ae, model_ae))

        print("AE val embedding:", z_va.shape)

    X_tr = tr_df[feature_cols_s1]
    X_va = va_df[feature_cols_s1]

    y_tr = y_all[tr_idx]
    y_va = y_all[va_idx]
    sw_tr = sw_all[tr_idx]

    m_lgb = lgb.LGBMRegressor(**lgb_params_s1)
    m_lgb.fit(
        X_tr, y_tr,
        sample_weight=sw_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="mae",
        callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(-1)]
    )
    oof_s1_lgb[va_idx] = from_train_pred(m_lgb.predict(X_va))
    models_s1_lgb.append(m_lgb)

    try:
        m_xgb = xgb.XGBRegressor(**xgb_params_s1)
        m_xgb.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], verbose=False)
    except Exception:
        xgb_fb = dict(xgb_params_s1)
        xgb_fb["objective"] = "reg:squarederror"
        m_xgb = xgb.XGBRegressor(**xgb_fb)
        m_xgb.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], verbose=False)

    oof_s1_xgb[va_idx] = from_train_pred(m_xgb.predict(X_va))
    models_s1_xgb.append(m_xgb)

    m_cat = cb.CatBoostRegressor(**cat_params_s1)
    m_cat.fit(
        X_tr, y_tr,
        sample_weight=sw_tr,
        eval_set=(X_va, y_va),
        verbose=False,
        use_best_model=True
    )
    oof_s1_cat[va_idx] = from_train_pred(m_cat.predict(X_va))
    models_s1_cat.append(m_cat)

    avg3 = (oof_s1_lgb[va_idx] + oof_s1_xgb[va_idx] + oof_s1_cat[va_idx]) / 3
    print("Fold Stage1 avg MAE:", mae(y_raw[va_idx], avg3))

mae_s1_lgb = mae(y_raw, oof_s1_lgb)
mae_s1_xgb = mae(y_raw, oof_s1_xgb)
mae_s1_cat = mae(y_raw, oof_s1_cat)

print("\nStage1 single OOF")
print("LGB:", mae_s1_lgb)
print("XGB:", mae_s1_xgb)
print("CAT:", mae_s1_cat)

s1_maes = {"lgb": mae_s1_lgb, "xgb": mae_s1_xgb, "cat": mae_s1_cat}
s1_oof_by = {"lgb": oof_s1_lgb, "xgb": oof_s1_xgb, "cat": oof_s1_cat}

best_s1_models = None
best_s1_p = None
best_s1_mae = float("inf")

for models in powerset_models():
    for p in JOINT_P_GRID:
        pred_tmp, _, _ = _ensemble_pred(s1_oof_by, s1_maes, models, p)
        m = mae(y_raw, pred_tmp)
        if m < best_s1_mae:
            best_s1_mae = m
            best_s1_models = models
            best_s1_p = p

oof_s1, w_s1, ws_s1 = _ensemble_pred(s1_oof_by, s1_maes, best_s1_models, best_s1_p)
s1_mae = mae(y_raw, oof_s1)

print("Best Stage1:", best_s1_models, best_s1_p, s1_mae)
print("Stage1 done:", elapsed(t0))


# ============================================================
# 8. AE test embedding
# ============================================================

if USE_AUTOENCODER and ae_fold_artifacts:
    z_folds = []

    for med_ae, sc_ae, model_ae in ae_fold_artifacts:
        X_te_raw = ae_prepare_matrix(test, ae_input_cols, med_ae)
        z_te = ae_encode_matrix(X_te_raw, sc_ae, model_ae, AE_DEVICE)
        z_folds.append(z_te)

    z_mean = np.mean(z_folds, axis=0)

    for i, c in enumerate(AE_COLS):
        test[c] = z_mean[:, i]

    print("AE test embeddings:", z_mean.shape)


# ============================================================
# 9. Stage1 prediction lag features
# ============================================================

PRED_LAG_COLS = [
    "pred_lag1", "pred_lag2", "pred_lag3",
    "pred_diff1", "pred_diff2",
    "pred_roll3_mean", "pred_roll5_mean",
    "pred_ewm3",
    "pred_lag1_log"
]

def add_pred_lag_features(df, pred_vec, fill_value):
    df = df.sort_values(["scenario_id", "timeslot", "ID"]).reset_index(drop=True)
    p = np.asarray(pred_vec, dtype=np.float64)

    if len(p) != len(df):
        raise ValueError("pred_vec length mismatch")

    df["_pred_for_lag"] = p

    g = df.groupby("scenario_id")["_pred_for_lag"]

    df["pred_lag1"] = g.shift(1)
    df["pred_lag2"] = g.shift(2)
    df["pred_lag3"] = g.shift(3)

    s1 = g.shift(1)
    s2 = g.shift(2)
    s3 = g.shift(3)

    df["pred_diff1"] = (s1 - s2).fillna(0)
    df["pred_diff2"] = (s2 - s3).fillna(0)

    df["pred_roll3_mean"] = g.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    df["pred_roll5_mean"] = g.transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    df["pred_ewm3"] = g.transform(lambda x: x.shift(1).ewm(alpha=0.3, adjust=False).mean())

    df["pred_lag1_log"] = np.log1p(df["pred_lag1"].clip(lower=0))

    for c in PRED_LAG_COLS:
        df[c] = df[c].fillna(fill_value)

    df = df.drop(columns=["_pred_for_lag"])

    return df


train = add_pred_lag_features(train, oof_s1, global_mean)


# ============================================================
# 10. Stage 2 models
# ============================================================

feature_cols_s2 = feature_cols_s1 + PRED_LAG_COLS

lgb_params_s2 = dict(
    objective="regression_l1",
    n_estimators=12000,
    learning_rate=0.02,
    max_depth=-1,
    num_leaves=511,
    min_child_samples=80,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.65,
    reg_alpha=0.15,
    reg_lambda=4.0,
    random_state=SEED,
    verbose=-1
)

xgb_params_s2 = dict(
    objective="reg:absoluteerror",
    n_estimators=8000,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.65,
    colsample_bynode=0.5,
    reg_alpha=0.15,
    reg_lambda=2.5,
    random_state=SEED,
    tree_method="hist",
    eval_metric="mae",
    early_stopping_rounds=120,
    verbosity=0
)

cat_params_s2 = dict(
    iterations=8000,
    learning_rate=0.02,
    depth=6,
    l2_leaf_reg=4.0,
    bootstrap_type="MVS",
    subsample=0.8,
    colsample_bylevel=0.65,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=SEED,
    task_type="CPU",
    early_stopping_rounds=120
)

section("Stage 2 - pred-lag stack")
t0 = time.time()

oof_s2_lgb = np.zeros(len(train))
oof_s2_xgb = np.zeros(len(train))
oof_s2_cat = np.zeros(len(train))

models_s2_lgb = []
models_s2_xgb = []
models_s2_cat = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(train, kf_y, groups=groups), 1):
    print(f"\n--- Stage2 Fold {fold} ---")

    tr_df = train.iloc[tr_idx].copy()
    va_df = train.iloc[va_idx].copy()

    X_tr = tr_df[feature_cols_s2]
    X_va = va_df[feature_cols_s2]

    y_tr = y_all[tr_idx]
    y_va = y_all[va_idx]
    sw_tr = sw_all[tr_idx]

    m_lgb = lgb.LGBMRegressor(**lgb_params_s2)
    m_lgb.fit(
        X_tr, y_tr,
        sample_weight=sw_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="mae",
        callbacks=[lgb.early_stopping(120, verbose=False), lgb.log_evaluation(-1)]
    )
    oof_s2_lgb[va_idx] = from_train_pred(m_lgb.predict(X_va))
    models_s2_lgb.append(m_lgb)

    try:
        m_xgb = xgb.XGBRegressor(**xgb_params_s2)
        m_xgb.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], verbose=False)
    except Exception:
        xgb_fb = dict(xgb_params_s2)
        xgb_fb["objective"] = "reg:squarederror"
        m_xgb = xgb.XGBRegressor(**xgb_fb)
        m_xgb.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], verbose=False)

    oof_s2_xgb[va_idx] = from_train_pred(m_xgb.predict(X_va))
    models_s2_xgb.append(m_xgb)

    m_cat = cb.CatBoostRegressor(**cat_params_s2)
    m_cat.fit(
        X_tr, y_tr,
        sample_weight=sw_tr,
        eval_set=(X_va, y_va),
        verbose=False,
        use_best_model=True
    )
    oof_s2_cat[va_idx] = from_train_pred(m_cat.predict(X_va))
    models_s2_cat.append(m_cat)

    avg3 = (oof_s2_lgb[va_idx] + oof_s2_xgb[va_idx] + oof_s2_cat[va_idx]) / 3

    print("LGB:", mae(y_raw[va_idx], oof_s2_lgb[va_idx]))
    print("XGB:", mae(y_raw[va_idx], oof_s2_xgb[va_idx]))
    print("CAT:", mae(y_raw[va_idx], oof_s2_cat[va_idx]))
    print("AVG:", mae(y_raw[va_idx], avg3))

mae_s2_lgb = mae(y_raw, oof_s2_lgb)
mae_s2_xgb = mae(y_raw, oof_s2_xgb)
mae_s2_cat = mae(y_raw, oof_s2_cat)

print("\nStage2 single OOF")
print("LGB:", mae_s2_lgb)
print("XGB:", mae_s2_xgb)
print("CAT:", mae_s2_cat)

s2_maes = {"lgb": mae_s2_lgb, "xgb": mae_s2_xgb, "cat": mae_s2_cat}
s2_oof_by = {"lgb": oof_s2_lgb, "xgb": oof_s2_xgb, "cat": oof_s2_cat}

best_s2_models = None
best_s2_p = None
best_s2_mae = float("inf")

for models in powerset_models():
    for p in JOINT_P_GRID:
        pred_tmp, _, _ = _ensemble_pred(s2_oof_by, s2_maes, models, p)
        m = mae(y_raw, pred_tmp)
        if m < best_s2_mae:
            best_s2_mae = m
            best_s2_models = models
            best_s2_p = p

oof_s2_ens, w_s2, ws_s2 = _ensemble_pred(s2_oof_by, s2_maes, best_s2_models, best_s2_p)
s2_ens_mae = mae(y_raw, oof_s2_ens)

print("Best Stage2:", best_s2_models, best_s2_p, s2_ens_mae)
print("Stage2 done:", elapsed(t0))


# ============================================================
# 11. Final joint search
# ============================================================

section("Final joint search")

s2_candidates = {
    "lgb": oof_s2_lgb,
    "xgb": oof_s2_xgb,
    "cat": oof_s2_cat
}

s2_mae_ref = {
    "lgb": mae_s2_lgb,
    "xgb": mae_s2_xgb,
    "cat": mae_s2_cat
}

alpha_grid = np.arange(0.0, 1.0 + 1e-12, FINAL_ALPHA_STEP)

best_final = {
    "mae": float("inf"),
    "models": None,
    "p": None,
    "alpha": None,
    "clip_q": None,
    "oof_s2": None
}

for models in powerset_models():
    for p in JOINT_P_GRID:
        pred_s2, _, _ = _ensemble_pred(s2_candidates, s2_mae_ref, models, p)

        for alpha in alpha_grid:
            blend = alpha * oof_s1 + (1 - alpha) * pred_s2

            for q in FINAL_CLIP_Q_GRID:
                hi = float(np.percentile(y_raw, 100 * q))
                pred = np.minimum(np.maximum(blend, CLIP_PRED_MIN), hi)
                m = mae(y_raw, pred)

                if m < best_final["mae"]:
                    best_final.update({
                        "mae": float(m),
                        "models": list(models),
                        "p": float(p),
                        "alpha": float(alpha),
                        "clip_q": float(q),
                        "oof_s2": pred_s2.copy()
                    })

best_s2_models = best_final["models"]
best_s2_p = best_final["p"]
best_alpha = best_final["alpha"]
best_clip_q = best_final["clip_q"]
oof_s2_used = best_final["oof_s2"]

blend_oof = best_alpha * oof_s1 + (1 - best_alpha) * oof_s2_used
hi = float(np.percentile(y_raw, 100 * best_clip_q))
final_joint_oof = np.minimum(np.maximum(blend_oof, CLIP_PRED_MIN), hi)

joint_mae = mae(y_raw, final_joint_oof)

print("Joint best")
print("models:", best_s2_models)
print("p:", best_s2_p)
print("alpha:", best_alpha)
print("clip_q:", best_clip_q)
print("joint MAE:", joint_mae)


# ============================================================
# 12. Advanced finalization: timeslot alpha + meta ridge
# ============================================================

section("Advanced finalization")

final_methods = {"joint": joint_mae}
final_oof_candidates = {"joint": final_joint_oof}

# timeslot alpha
ts_alpha_map = {}

if ENABLE_TIMESLOT_ALPHA_SEARCH and "timeslot" in train.columns:
    ts_arr = train["timeslot"].values
    ts_oof = np.zeros(len(train))

    alpha_grid_ts = np.arange(0.0, 1.0 + 1e-12, TIMESLOT_ALPHA_STEP)

    for ts in np.unique(ts_arr):
        idx = ts_arr == ts
        y_ts = y_raw[idx]
        s1_ts = oof_s1[idx]
        s2_ts = oof_s2_used[idx]

        best_a = best_alpha
        best_m = float("inf")

        for a in alpha_grid_ts:
            pred_ts = a * s1_ts + (1 - a) * s2_ts
            m = np.mean(np.abs(y_ts - pred_ts))
            if m < best_m:
                best_m = m
                best_a = a

        a_smooth = (1 - TIMESLOT_ALPHA_SMOOTH_TO_GLOBAL) * best_a + TIMESLOT_ALPHA_SMOOTH_TO_GLOBAL * best_alpha
        ts_alpha_map[int(ts)] = float(a_smooth)
        ts_oof[idx] = a_smooth * s1_ts + (1 - a_smooth) * s2_ts

    hi_ts = float(np.percentile(y_raw, 100 * best_clip_q))
    ts_oof = np.minimum(np.maximum(ts_oof, CLIP_PRED_MIN), hi_ts)

    ts_mae = mae(y_raw, ts_oof)
    final_methods["timeslot_alpha"] = ts_mae
    final_oof_candidates["timeslot_alpha"] = ts_oof

    print("timeslot alpha MAE:", ts_mae)


# meta ridge
meta_fold_models = []

if ENABLE_OOF_META_STACK:
    ts = train["timeslot"].values.astype(float) if "timeslot" in train.columns else np.zeros(len(train))
    ang = 2.0 * np.pi * ts / 24.0

    X_meta = np.column_stack([
        oof_s1,
        oof_s2_lgb,
        oof_s2_xgb,
        oof_s2_cat,
        oof_s2_used,
        np.sin(ang),
        np.cos(ang)
    ])

    oof_meta = np.zeros(len(train))

    for fold, (tr_idx, va_idx) in enumerate(kf.split(train, kf_y, groups=groups), 1):
        m = RidgeCV(alphas=META_ALPHAS, fit_intercept=True)
        m.fit(X_meta[tr_idx], y_raw[tr_idx])
        oof_meta[va_idx] = m.predict(X_meta[va_idx])
        meta_fold_models.append(m)

    hi_meta = float(np.percentile(y_raw, 100 * best_clip_q))
    oof_meta = np.minimum(np.maximum(oof_meta, CLIP_PRED_MIN), hi_meta)

    meta_mae = mae(y_raw, oof_meta)
    final_methods["meta_ridge"] = meta_mae
    final_oof_candidates["meta_ridge"] = oof_meta

    print("meta ridge MAE:", meta_mae)


FINAL_METHOD = min(final_methods, key=final_methods.get)
final_oof = final_oof_candidates[FINAL_METHOD]
best_final_mae = final_methods[FINAL_METHOD]

print("\n===== Final method selected =====")
print("method:", FINAL_METHOD)
print("OOF MAE:", best_final_mae)


# optional global calibration
if GLOBAL_CALIBRATION_SEARCH:
    calib_params, calib_mae, calib_df = search_scale_bias_clip(final_oof, y_raw)

    if calib_mae < best_final_mae:
        final_oof, _ = apply_scale_bias_clip(
            final_oof,
            calib_params["scale"],
            calib_params["bias"],
            calib_params["clip_q"]
        )
        best_final_mae = calib_mae
        USE_GLOBAL_CALIBRATION = True
    else:
        USE_GLOBAL_CALIBRATION = False
        calib_params = {
            "scale": 1.0,
            "bias": 0.0,
            "clip_q": None
        }
else:
    USE_GLOBAL_CALIBRATION = False
    calib_params = {
        "scale": 1.0,
        "bias": 0.0,
        "clip_q": None
    }

print("\n===== Final OOF =====")
print("FINAL_METHOD:", FINAL_METHOD)
print("USE_GLOBAL_CALIBRATION:", USE_GLOBAL_CALIBRATION)
print("calib_params:", calib_params)
print("FINAL OOF MAE:", best_final_mae)


# ============================================================
# 13. Predict Test
# ============================================================

section("Predict test")

test = test.sort_values(["scenario_id", "timeslot", "ID"]).reset_index(drop=True)

X_test_s1 = test[feature_cols_s1]

p_lgb_s1 = np.mean([from_train_pred(m.predict(X_test_s1)) for m in models_s1_lgb], axis=0)
p_xgb_s1 = np.mean([from_train_pred(m.predict(X_test_s1)) for m in models_s1_xgb], axis=0)
p_cat_s1 = np.mean([from_train_pred(m.predict(X_test_s1)) for m in models_s1_cat], axis=0)

s1_test_by = {
    "lgb": p_lgb_s1,
    "xgb": p_xgb_s1,
    "cat": p_cat_s1
}

pred_s1_test = sum(w_s1[m] * s1_test_by[m] for m in best_s1_models) / ws_s1

test = add_pred_lag_features(test, pred_s1_test, global_mean)

X_test_s2 = test[feature_cols_s2]

pred_s2_lgb_test = np.mean([from_train_pred(m.predict(X_test_s2)) for m in models_s2_lgb], axis=0)
pred_s2_xgb_test = np.mean([from_train_pred(m.predict(X_test_s2)) for m in models_s2_xgb], axis=0)
pred_s2_cat_test = np.mean([from_train_pred(m.predict(X_test_s2)) for m in models_s2_cat], axis=0)

s2_test_by = {
    "lgb": pred_s2_lgb_test,
    "xgb": pred_s2_xgb_test,
    "cat": pred_s2_cat_test
}

# recompute selected s2 weights
s2_selected_maes = {
    "lgb": mae_s2_lgb,
    "xgb": mae_s2_xgb,
    "cat": mae_s2_cat
}

w_s2_final = {m: 1.0 / (s2_selected_maes[m] ** best_s2_p) for m in best_s2_models}
ws_s2_final = sum(w_s2_final.values())

pred_s2_test = sum(w_s2_final[m] * s2_test_by[m] for m in best_s2_models) / ws_s2_final

pred_final_base = best_alpha * pred_s1_test + (1 - best_alpha) * pred_s2_test

if FINAL_METHOD == "timeslot_alpha" and len(ts_alpha_map) > 0:
    ts_test = test["timeslot"].map(ts_alpha_map).fillna(best_alpha).values.astype(float)
    pred_final_base = ts_test * pred_s1_test + (1 - ts_test) * pred_s2_test
    print("test final method: timeslot_alpha")

elif FINAL_METHOD == "meta_ridge" and len(meta_fold_models) > 0:
    tsv = test["timeslot"].values.astype(float) if "timeslot" in test.columns else np.zeros(len(test))
    ang = 2.0 * np.pi * tsv / 24.0

    X_meta_test = np.column_stack([
        pred_s1_test,
        pred_s2_lgb_test,
        pred_s2_xgb_test,
        pred_s2_cat_test,
        pred_s2_test,
        np.sin(ang),
        np.cos(ang)
    ])

    pred_final_base = np.mean([m.predict(X_meta_test) for m in meta_fold_models], axis=0)
    print("test final method: meta_ridge")
else:
    print("test final method: joint")

pred = np.maximum(pred_final_base, CLIP_PRED_MIN)

hi = float(np.percentile(y_raw, 100 * best_clip_q))
pred = np.minimum(pred, hi)

if USE_GLOBAL_CALIBRATION:
    pred, _ = apply_scale_bias_clip(
        pred,
        calib_params["scale"],
        calib_params["bias"],
        calib_params["clip_q"]
    )

print(pd.Series(pred).describe())


# ============================================================
# 14. Save submission + diagnostics
# ============================================================

submission = pd.DataFrame({
    "ID": test["ID"],
    TARGET: pred
})

output_path = "submission_hybrid_h2_autoencoder_stage2_meta_calib.csv"
submission.to_csv(output_path, index=False)

oof_path = "oof_hybrid_h2_autoencoder_stage2_meta_calib.csv"
pd.DataFrame({
    "ID": train["ID"],
    "y_true": y_raw,
    "oof_pred": final_oof,
    "abs_error": np.abs(y_raw - final_oof)
}).to_csv(oof_path, index=False)

print("저장 완료:", output_path)
print("OOF 저장 완료:", oof_path)
print("FINAL_METHOD:", FINAL_METHOD)
print("FINAL OOF MAE:", best_final_mae)
print("calib params:", calib_params)
display(submission.head())
print(submission[TARGET].describe())

train shape: (250000, 94)
test shape : (50000, 93)
time feature 기준: scenario_id 내부 원본 행 순서
time feature 기준: scenario_id 내부 원본 행 순서
after feature train shape: (250000, 590)
after feature test shape : (50000, 589)
CV: StratifiedGroupKFold
Stage1 features: 620
AE cols: 32
sample weight mean/max: 1.0494846 1.375

  Stage 1 - LGB/XGB/CAT + AutoEncoder

--- Stage1 Fold 1 ---
AE val embedding: (83350, 32)
Fold Stage1 avg MAE: 8.570858755026405

--- Stage1 Fold 2 ---
AE val embedding: (83325, 32)
Fold Stage1 avg MAE: 8.57692086652707

--- Stage1 Fold 3 ---
AE val embedding: (83325, 32)
Fold Stage1 avg MAE: 8.86077674066525

Stage1 single OOF
LGB: 8.694476596571366
XGB: 8.739900947252657
CAT: 8.708656953893971
Best Stage1: ['lgb', 'cat'] 4.0 8.661223283714984
Stage1 done: 249m 50s
AE test embeddings: (50000, 32)

  Stage 2 - pred-lag stack

--- Stage2 Fold 1 ---
LGB: 8.800433927297815
XGB: 8.759890170219219
CAT: 8.758433453042977
AVG: 8.7486561002074

--- Stage2 Fold 2 ---
LGB: 8.76294578465406

,scale,bias,clip_q,clip_upper,mae
995,1.030,-0.25,NaN,NaN,8.643475
941,1.025,-0.25,NaN,NaN,8.644141
887,1.020,-0.25,NaN,NaN,8.645415
893,1.020,0.00,NaN,NaN,8.645540
839,1.015,0.00,NaN,NaN,8.645591
947,1.025,0.00,NaN,NaN,8.646092
785,1.010,0.00,NaN,NaN,8.646224
1001,1.030,0.00,NaN,NaN,8.647232
833,1.015,-0.25,NaN,NaN,8.647288
731,1.005,0.00,NaN,NaN,8.647444



===== Final OOF =====
FINAL_METHOD: timeslot_alpha
USE_GLOBAL_CALIBRATION: True
calib_params: {'scale': 1.03, 'bias': -0.25, 'clip_q': None, 'clip_upper': None}
FINAL OOF MAE: 8.643474503339831

  Predict test
test final method: timeslot_alpha
count    50000.000000
mean        19.128117
std         14.298690
min          0.001408
25%          5.294781
50%         14.281502
75%         34.157779
max         74.280569
dtype: float64
저장 완료: submission_hybrid_h2_autoencoder_stage2_meta_calib.csv
OOF 저장 완료: oof_hybrid_h2_autoencoder_stage2_meta_calib.csv
FINAL_METHOD: timeslot_alpha
FINAL OOF MAE: 8.643474503339831
calib params: {'scale': 1.03, 'bias': -0.25, 'clip_q': None, 'clip_upper': None}


,ID,avg_delay_minutes_next_30m
0,TEST_015375,16.770802
1,TEST_015376,28.019603
2,TEST_015377,33.089138
3,TEST_015378,35.242642
4,TEST_015379,35.973944


count    50000.000000
mean        19.128117
std         14.298690
min          0.001408
25%          5.294781
50%         14.281502
75%         34.157779
max         74.280569
Name: avg_delay_minutes_next_30m, dtype: float64
